In [2]:
!pip install -q tensorflow
import tensorflow as tf
import pandas as pd
import numpy as np
import os
import pickle
import shutil
from IPython.display import FileLink
print(tf.__version__)

2025-08-10 21:31:36.205113: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1754861496.502067      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1754861496.582787      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


2.18.0


# **Importing the Data:**

In [3]:
#Kaggle Import of the Data



csv_path = "/kaggle/input/energy-production/Gesamterzeugung_hourly.csv"
#neu = pd.read_csv(csv_path, index_col='Datum', parse_dates=True)
#display(neu)
df=neu = pd.read_csv(csv_path, index_col='Datum', parse_dates=True)
display(df)


,Stromerzeugung Gesamt
Datum,
2015-01-01 00:00:00,51238.75
2015-01-01 01:00:00,49749.00
2015-01-01 02:00:00,49014.75
2015-01-01 03:00:00,47954.75
2015-01-01 04:00:00,48187.00
...,...
2018-12-31 19:00:00,55441.00
2018-12-31 20:00:00,54499.50
2018-12-31 21:00:00,54526.75


# **Defining Functions**

In [4]:
def create_preprocessing_datasets(df, seq_length, batch_size, prediction_horizon):
    """
    Creates train and validation datasets with the given parameters
    
    Args:
        df: DataFrame with electricity generation data
        seq_length: Length of input sequences (default: 336)
        batch_size: Batch size for training (default: 64)
        prediction_horizon: Number of timesteps to predict (default: 24)
    
    Returns:
        train_ds, valid_ds: TensorFlow datasets for training and validation
    """
    # Prepare data for normalization
    Strom_train = df["Stromerzeugung Gesamt"]["2015-01-01 00:00:00":"2018-06-30 23:00:00"]
    mean = Strom_train.mean()
    std = Strom_train.std()
    
    # Normalize training and validation data
    Strom_train = (((df["Stromerzeugung Gesamt"]["2015-01-01 00:00:00":"2018-06-30 23:00:00"]) - mean)/std)
    Strom_valid = (((df["Stromerzeugung Gesamt"]["2018-07-01 00:00:00":"2018-12-31 23:00:00"]) - mean)/std)
    
    # Define function to split sequences into inputs and targets
    def split_inputs_and_targets(time_series, ahead=prediction_horizon):
        return time_series[:, :-ahead], time_series[:, -ahead:]
    
    # Create training dataset
    train_ds = tf.keras.utils.timeseries_dataset_from_array(
        Strom_train.to_numpy(),
        targets=None,
        sequence_length=seq_length + prediction_horizon,
        batch_size=batch_size,
        shuffle=False,
        seed=42
    ).map(split_inputs_and_targets)

    # Create validation dataset
    valid_ds = tf.keras.utils.timeseries_dataset_from_array(
        Strom_valid.to_numpy(),
        targets=None,
        sequence_length=seq_length + prediction_horizon,
        batch_size=batch_size
    ).map(split_inputs_and_targets)
    
    return train_ds, valid_ds


In [5]:
def create_model(model_type, layers=1, prediction_horizon=24):
    """
    Creates a Keras model with the specified parameters
    
    Args:
        model_type: 'GRU' or 'LSTM'
        layers: Number of layers (1 or 2)
        prediction_horizon: Output dimension (1 for 1h, 24 for 24h)
    
    Returns:
        compiled model
    """
    tf.random.set_seed(42)
    
    # Choose layer type
    if model_type.upper() == 'GRU':
        layer_class = tf.keras.layers.GRU
    elif model_type.upper() == 'LSTM':
        layer_class = tf.keras.layers.LSTM
    else:
        raise ValueError("model_type must be 'GRU' or 'LSTM'")
    
    # Build model architecture with combined logic
    model_layers = [tf.keras.layers.Input(shape=(None, 1))]
    
    if layers == 1:
        # Single layer with 64 units
        model_layers.append(layer_class(64, dropout=0.1))
    elif layers == 2:
        # Two layers with 32 units each
        model_layers.append(layer_class(32, 
                                      dropout=0.1,
                                      recurrent_dropout=0.3,
                                      return_sequences=True))
        model_layers.append(layer_class(32, dropout=0.1))
    else:
        raise ValueError("layers must be 1 or 2")
    
    # Output layer
    model_layers.append(tf.keras.layers.Dense(prediction_horizon))
    
    # Create model
    model = tf.keras.Sequential(model_layers)
    
    # Optimizer with fixed learning rate
    opt = tf.keras.optimizers.Adam(
        learning_rate=0.005,
        clipvalue=1.0
    )
    
    # Compile model
    model.compile(
        loss=tf.keras.losses.Huber(),
        optimizer=opt,
        metrics=["mae"]
    )
    
    return model


In [6]:
def train_model(model, model_name, epochs=100):
    """
    Trains a model with callbacks and saves checkpoints and history
    
    Args:
        model: Compiled Keras model
        model_name: Name for the model (used for folder and file names)
        epochs: Number of training epochs (default: 100)
    
    Returns:
        history: Training history object
    """
    
    # Fixed values
    base_dir = "./models/"
    # train_ds and valid_ds are used directly from the global scope
    
    # Create model directory
    model_dir = os.path.join(base_dir, model_name)
    os.makedirs(model_dir, exist_ok=True)
    
    # File paths
    model_path = os.path.join(model_dir, f"{model_name}-epoch-{{epoch:02d}}.keras")
    history_path = os.path.join(model_dir, f"{model_name}_history.pkl")
    
    # Callbacks
    lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        verbose=1,
        min_lr=1e-6
    )
    
    model_checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(
        filepath=model_path,
        save_best_only=False,
        save_freq="epoch",
        verbose=1
    )
    
    # Training
    history = model.fit(
        train_ds,
        validation_data=valid_ds,
        epochs=epochs,
        callbacks=[model_checkpoint_cb, lr_scheduler]
    )
    
    # Save history
    with open(history_path, 'wb') as file:
        pickle.dump(history.history, file)
    
    # Create ZIP for download
    shutil.make_archive(model_name, 'zip', model_dir)
    
    # Print info
    print(f"\nGespeicherte Dateien:")
    print(f"- Modell-Checkpoints: {model_dir}/*.keras")
    print(f"- Trainingshistorie: {history_path}")
    print(f"- ZIP-Datei: {model_name}.zip")
    
    # Display download link (for Kaggle)
    try:
        display(FileLink(f"{model_name}.zip"))
    except:
        print(f"Download-Link: {model_name}.zip")
    
    return history


# **Training of the Models**

## 1Layer - 24h Prediction 

In [7]:
#model_GRU64_gradclip_RedPl
import tensorflow as tf

tf.random.set_seed(42)

# Create datasets (using preprocessing function)
train_ds, valid_ds = create_preprocessing_datasets(df, 336, 64, 24)

# Create model (using model function)
model = create_model('GRU', layers=1, prediction_horizon=24)

# Train model (using training function)
history = train_model(model, "model_GRU64_gradclip_RedPl", epochs=100)


Epoch 1/100


I0000 00:00:1754848552.837280     106 cuda_dnn.cc:529] Loaded cuDNN version 90300


472/474 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.2357 - mae: 0.5546
Epoch 1: saving model to ./models/model_GRU64_gradclip_RedPl/model_GRU64_gradclip_RedPl-epoch-01.keras
474/474 ━━━━━━━━━━━━━━━━━━━━ 11s 15ms/step - loss: 0.2355 - mae: 0.5543 - val_loss: 0.2303 - val_mae: 0.5507 - learning_rate: 0.0050
Epoch 2/100
472/474 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.1680 - mae: 0.4489
Epoch 2: saving model to ./models/model_GRU64_gradclip_RedPl/model_GRU64_gradclip_RedPl-epoch-02.keras
474/474 ━━━━━━━━━━━━━━━━━━━━ 7s 14ms/step - loss: 0.1680 - mae: 0.4488 - val_loss: 0.2159 - val_mae: 0.5280 - learning_rate: 0.0050
Epoch 3/100
470/474 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.1582 - mae: 0.4309
Epoch 3: saving model to ./models/model_GRU64_gradclip_RedPl/model_GRU64_gradclip_RedPl-epoch-03.keras
474/474 ━━━━━━━━━━━━━━━━━━━━ 7s 14ms/step - loss: 0.1581 - mae: 0.4308 - val_loss: 0.2905 - val_mae: 0.6237 - learning_rate: 0.0050
Epoch 4/100
474/474 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - l

/kaggle/working/model_GRU64_gradclip_RedPl.zip

In [8]:
#model_LSTM64_gradclip_RedPl

import tensorflow as tf

tf.random.set_seed(42)

# Create datasets (using preprocessing function)
train_ds, valid_ds = create_preprocessing_datasets(df, 336, 64, 24)

# Create model (using model function)
model = create_model('LSTM', layers=1, prediction_horizon=24)

# Train model (using training function)
history = train_model(model, "model_LSTM64_gradclip_RedPl", epochs=100)

Epoch 1/100
474/474 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.2261 - mae: 0.5416
Epoch 1: saving model to ./models/model_LSTM64_gradclip_RedPl/model_LSTM64_gradclip_RedPl-epoch-01.keras
474/474 ━━━━━━━━━━━━━━━━━━━━ 9s 15ms/step - loss: 0.2261 - mae: 0.5415 - val_loss: 0.3209 - val_mae: 0.6712 - learning_rate: 0.0050
Epoch 2/100
472/474 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.2333 - mae: 0.5497
Epoch 2: saving model to ./models/model_LSTM64_gradclip_RedPl/model_LSTM64_gradclip_RedPl-epoch-02.keras
474/474 ━━━━━━━━━━━━━━━━━━━━ 7s 14ms/step - loss: 0.2332 - mae: 0.5495 - val_loss: 0.2323 - val_mae: 0.5533 - learning_rate: 0.0050
Epoch 3/100
471/474 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.1736 - mae: 0.4613
Epoch 3: saving model to ./models/model_LSTM64_gradclip_RedPl/model_LSTM64_gradclip_RedPl-epoch-03.keras
474/474 ━━━━━━━━━━━━━━━━━━━━ 7s 14ms/step - loss: 0.1735 - mae: 0.4612 - val_loss: 0.2625 - val_mae: 0.5930 - learning_rate: 0.0050
Epoch 4/100
470/474 ━━━━━━━━━━━━━━━━━━━━

/kaggle/working/model_LSTM64_gradclip_RedPl.zip

## 1Layer - 1h Prediction 

In [13]:
#model_1Hour_GRU64_gradclip_RedPl_seq336

tf.random.set_seed(42)

# Create datasets (using preprocessing function)
train_ds, valid_ds = create_preprocessing_datasets(df, seq_length=336, batch_size=64, prediction_horizon=1)

# Create model (using model function)
model = create_model('GRU', layers=1, prediction_horizon=1)

# Train model (using training function)
history = train_model(model, "model_1Hour_GRU64_gradclip_RedPl_seq336", epochs=100)

Epoch 1/100
473/474 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0701 - mae: 0.2404
Epoch 1: saving model to ./models/model_1Hour_GRU64_gradclip_RedPl_seq336/model_1Hour_GRU64_gradclip_RedPl_seq336-epoch-01.keras
474/474 ━━━━━━━━━━━━━━━━━━━━ 9s 16ms/step - loss: 0.0701 - mae: 0.2403 - val_loss: 0.0564 - val_mae: 0.1903 - learning_rate: 0.0050
Epoch 2/100
472/474 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0531 - mae: 0.1865
Epoch 2: saving model to ./models/model_1Hour_GRU64_gradclip_RedPl_seq336/model_1Hour_GRU64_gradclip_RedPl_seq336-epoch-02.keras
474/474 ━━━━━━━━━━━━━━━━━━━━ 6s 14ms/step - loss: 0.0531 - mae: 0.1865 - val_loss: 0.0627 - val_mae: 0.2194 - learning_rate: 0.0050
Epoch 3/100
473/474 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0560 - mae: 0.1910
Epoch 3: saving model to ./models/model_1Hour_GRU64_gradclip_RedPl_seq336/model_1Hour_GRU64_gradclip_RedPl_seq336-epoch-03.keras
474/474 ━━━━━━━━━━━━━━━━━━━━ 6s 14ms/step - loss: 0.0560 - mae: 0.1909 - val_loss: 0.0620 - val_mae:

/kaggle/working/model_1Hour_GRU64_gradclip_RedPl_seq336.zip

In [14]:
#model_1Hour_LSTM64_gradclip_RedPl_seq336

tf.random.set_seed(42)

# Create datasets (using preprocessing function)
train_ds, valid_ds = create_preprocessing_datasets(df, seq_length=336, batch_size=64, prediction_horizon=1)

# Create model (using model function)
model = create_model('LSTM', layers=1, prediction_horizon=1)

# Train model (using training function)
history = train_model(model, "model_1Hour_LSTM64_gradclip_RedPl_seq336", epochs=100)

Epoch 1/100
472/474 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0860 - mae: 0.2858
Epoch 1: saving model to ./models/model_1Hour_LSTM64_gradclip_RedPl_seq336/model_1Hour_LSTM64_gradclip_RedPl_seq336-epoch-01.keras
474/474 ━━━━━━━━━━━━━━━━━━━━ 9s 15ms/step - loss: 0.0859 - mae: 0.2854 - val_loss: 0.0612 - val_mae: 0.2101 - learning_rate: 0.0050
Epoch 2/100
472/474 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0632 - mae: 0.2116
Epoch 2: saving model to ./models/model_1Hour_LSTM64_gradclip_RedPl_seq336/model_1Hour_LSTM64_gradclip_RedPl_seq336-epoch-02.keras
474/474 ━━━━━━━━━━━━━━━━━━━━ 7s 14ms/step - loss: 0.0633 - mae: 0.2119 - val_loss: 0.1098 - val_mae: 0.3523 - learning_rate: 0.0050
Epoch 3/100
472/474 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0811 - mae: 0.2858
Epoch 3: saving model to ./models/model_1Hour_LSTM64_gradclip_RedPl_seq336/model_1Hour_LSTM64_gradclip_RedPl_seq336-epoch-03.keras
474/474 ━━━━━━━━━━━━━━━━━━━━ 7s 14ms/step - loss: 0.0810 - mae: 0.2856 - val_loss: 0.0638 - va

/kaggle/working/model_1Hour_LSTM64_gradclip_RedPl_seq336.zip

## 2Layers - 24h Prediction 

In [7]:
#"model_GRU32x2_gradclip_RedPl_seq336_B128"

tf.random.set_seed(42)

# Create datasets (using preprocessing function)
train_ds, valid_ds = create_preprocessing_datasets(df, seq_length=336, batch_size=128, prediction_horizon=24)

# Create model (using model function)
model = create_model('GRU', layers=2, prediction_horizon=24)

# Train model (using training function)
history = train_model(model, "model_GRU32x2_gradclip_RedPl_seq336_B128", epochs=100)

Epoch 1/100
237/237 ━━━━━━━━━━━━━━━━━━━━ 0s 339ms/step - loss: 0.2860 - mae: 0.6230
Epoch 1: saving model to ./models/model_GRU32x2_gradclip_RedPl_seq336_B128/model_GRU32x2_gradclip_RedPl_seq336_B128-epoch-01.keras
237/237 ━━━━━━━━━━━━━━━━━━━━ 93s 357ms/step - loss: 0.2858 - mae: 0.6227 - val_loss: 0.2472 - val_mae: 0.5765 - learning_rate: 0.0050
Epoch 2/100
237/237 ━━━━━━━━━━━━━━━━━━━━ 0s 334ms/step - loss: 0.2127 - mae: 0.5229
Epoch 2: saving model to ./models/model_GRU32x2_gradclip_RedPl_seq336_B128/model_GRU32x2_gradclip_RedPl_seq336_B128-epoch-02.keras
237/237 ━━━━━━━━━━━━━━━━━━━━ 82s 347ms/step - loss: 0.2128 - mae: 0.5229 - val_loss: 0.2432 - val_mae: 0.5730 - learning_rate: 0.0050
Epoch 3/100
237/237 ━━━━━━━━━━━━━━━━━━━━ 0s 341ms/step - loss: 0.2062 - mae: 0.5116
Epoch 3: saving model to ./models/model_GRU32x2_gradclip_RedPl_seq336_B128/model_GRU32x2_gradclip_RedPl_seq336_B128-epoch-03.keras
237/237 ━━━━━━━━━━━━━━━━━━━━ 84s 354ms/step - loss: 0.2061 - mae: 0.5115 - val_loss: 0.

/kaggle/working/model_GRU32x2_gradclip_RedPl_seq336_B128.zip

In [ ]:
#"model_LSTM32x2_gradclip_RedPl_seq336_B128"

tf.random.set_seed(42)

# Create datasets (using preprocessing function)
train_ds, valid_ds = create_preprocessing_datasets(df, seq_length=336, batch_size=128, prediction_horizon=24)

# Create model (using model function)
model = create_model('LSTM', layers=2, prediction_horizon=24)

# Train model (using training function)
history = train_model(model, "model_LSTM32x2_gradclip_RedPl_seq336_B128", epochs=100)

## 2Layers - 1h Prediction 

In [8]:
#model_1Hour_GRU32x2_gradclip_RedPl_seq336_B128

tf.random.set_seed(42)

# Create datasets (using preprocessing function)
train_ds, valid_ds = create_preprocessing_datasets(df, seq_length=336, batch_size=128, prediction_horizon=24)

# Create model (using model function)
model = create_model('GRU', layers=2, prediction_horizon=1)

# Train model (using training function)
history = train_model(model, "#model_1Hour_GRU32x2_gradclip_RedPl_seq336_B128", epochs=100)

Epoch 1/100
 17/237 ━━━━━━━━━━━━━━━━━━━━ 1:27 398ms/step - loss: 0.2551 - mae: 0.5933

KeyboardInterrupt: 

In [ ]:
#model_1Hour_LSTM32x2_gradclip_RedPl_seq336_B128

tf.random.set_seed(42)

# Create datasets (using preprocessing function)
train_ds, valid_ds = create_preprocessing_datasets(df, seq_length=336, batch_size=128, prediction_horizon=24)

# Create model (using model function)
model = create_model('LSTM', layers=2, prediction_horizon=1)

# Train model (using training function)
history = train_model(model, "#model_1Hour_LSTM32x2_gradclip_RedPl_seq336_B128", epochs=100)